# Checkpoint 2: LSTM PyTorch Model (Multi-Class 5 categories)
This notebook implements the second phase of the Text Classification assignment using an RNN/LSTM model with PyTorch. The goal is to classify texts into Human, Google, Anthropic, Meta, and OpenAI.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import sys
import os
import matplotlib.pyplot as plt

# Adicionar a pasta raiz ao sys.path para conseguirmos importar o model_pytorch.py
sys.path.append(os.path.abspath('..'))

from models.model_pytorch import RNNClassifier, train_rnn_model, save_pytorch_model, load_pytorch_model, predict_pytorch


## 1. Carregar Dados
Nesta secção, carregamos os dados já preparados de treino, validação e teste.

In [ ]:
train_df = pd.read_csv('../data/dataset_treino.csv')
val_df = pd.read_csv('../data/dataset_validacao.csv')
test_df = pd.read_csv('../data/dataset_teste.csv')

print("Treino:", train_df.shape)
print("Validação:", val_df.shape)
print("Teste:", test_df.shape)

# As classes em 'source_code' já são 0, 1, 2, 3, 4.
X_train_raw = train_df['Text'].values
y_train = train_df['source_code'].values

X_val_raw = val_df['Text'].values
y_val = val_df['source_code'].values

print("Distribuição classes Treino:")
print(train_df['source_name'].value_counts())


## 2. Tokenização e Padding
Transformamos os textos em sequências de índices, com padding de tamanho fixo para alimentar a LSTM.

In [ ]:
from collections import Counter

all_words = ' '.join(X_train_raw).lower().split()
word_counts = Counter(all_words)

MAX_VOCAB = 10000
vocab = {word: i + 2 for i, (word, _) in enumerate(word_counts.most_common(MAX_VOCAB))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1
vocab_size = len(vocab)
print("Tamanho do Vocabulário:", vocab_size)

def tokenize_and_pad(texts, vocab, max_len=120):
    sequences = []
    for text in texts:
        tokens = text.lower().split()
        seq = [vocab.get(token, vocab['<UNK>']) for token in tokens[:max_len]]
        # Padding
        seq += [vocab['<PAD>']] * (max_len - len(seq))
        sequences.append(seq)
    return np.array(sequences)

MAX_LEN = 50

X_train_seq = tokenize_and_pad(X_train_raw, vocab, max_len=MAX_LEN)
X_val_seq = tokenize_and_pad(X_val_raw, vocab, max_len=MAX_LEN)

print("Shape de X_train_seq:", X_train_seq.shape)


## 3. Treino da LSTM PyTorch
Utilizar a função `train_rnn_model` adicionada no ficheiro `model_pytorch.py` pelo assistente.

In [ ]:
EPOCHS = 35
N_CLASSES = 5

trained_rnn = train_rnn_model(X_train_seq, y_train, vocab_size, n_classes=N_CLASSES, epochs=EPOCHS)


## 4. Avaliação no Conjunto de Validação
Avaliar o modelo treinado.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_pred_val = predict_pytorch(trained_rnn, X_val_seq, n_classes=N_CLASSES, is_rnn=True)

print("Acurácia Validação:", accuracy_score(y_val, y_pred_val))
print("\nReporte de Classificação Validação:\n", classification_report(y_val, y_pred_val, target_names=['Human', 'Google', 'Anthropic', 'Meta', 'OpenAI']))


## 5. Salvar o Modelo e Prever no Teste
A submissão deve processar os dados e guardar num csv o resultado.

In [ ]:
model_filepath = '../models/lstm_model.pt'
save_pytorch_model(trained_rnn, model_filepath)

X_test_raw = test_df['Text'].values
X_test_seq = tokenize_and_pad(X_test_raw, vocab, max_len=MAX_LEN)

y_pred_test = predict_pytorch(trained_rnn, X_test_seq, n_classes=N_CLASSES, is_rnn=True)

submission_df = test_df.copy()
submission_df['predicted_class'] = y_pred_test
source_mapping = {0: 'human', 1: 'google', 2: 'Anthropic', 3: 'meta', 4: 'openai'}
submission_df['predicted_source'] = submission_df['predicted_class'].map(source_mapping)

print("Algumas predições:\n", submission_df[['Text', 'predicted_source']].head())

submission_df.to_csv('../subm1/subm2-MIA-B.csv', index=False)
print("Ficheiro de submissão criado em subm1/subm2-MIA-B.csv")
